# Camera-query: finetune trên hai GPU Kaggle
Giữ Stage 1 (196), relevance và IRA; nén cùng camera thành 49 latent tokens. Chỉ finetune từ pretrained T5 hoặc resume checkpoint cùng profile camera_query. Không dùng checkpoint align/MVPruner/unpruned.

Seed 42; entropy weight 0.5; FP32; batch 16/process, application accumulation 2. Gradient checkpointing false, bật true chưa được hỗ trợ. Giữ nguyên batch khi lỗi OOM; đo VRAM thực tế trên Kaggle.

Prepared/cache phải khớp chính xác manifest v2, hash prepared files, model revision và index. Đường dẫn riêng cho profile này. Resume chỉ ở ranh giới epoch, cần checkpoint và đủ RNG hai rank; verify-resume kiểm tra tương thích trước launch nhưng probe một process không xác minh replay DDP. Runtime YAML riêng được lưu cùng archive.


In [ ]:
import os, json, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/huyvanzzz/Group_HND_car.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'huy-dinov2')
PREPARED_DATA_HF_REPO_ID = ''  # Dataset repo: manifest.json, train.jsonl, val.jsonl, test.jsonl at root.
CACHE_HF_REPO_ID = ''  # Separate DINO dataset repo: features.float16.memmap, index.json, manifest.json at root.
FINETUNE_CHECKPOINT_HF_REPO_ID = ''  # Optional model repo: checkpoint plus <filename>.rng/**.
FINETUNE_CHECKPOINT_FILENAME = 'finetune_latest.pt'
if PREPARED_DATA_HF_REPO_ID and PREPARED_DATA_HF_REPO_ID == CACHE_HF_REPO_ID:
    raise ValueError('Prepared data and DINO cache must use separate dataset repositories')
os.environ.update({'GITHUB_REPO_URL': GITHUB_REPO_URL, 'GITHUB_BRANCH': GITHUB_BRANCH,
                   'CONFIG': 'configs/dinov2_t5_efficient_mini_kaggle_2gpu_camera_query.yaml', 'PROFILE': 'outputs/dinov2_t5_efficient_mini_kaggle_2gpu_camera_query',
                   'PREPARED_DATA_HF_REPO_ID': PREPARED_DATA_HF_REPO_ID, 'CACHE_HF_REPO_ID': CACHE_HF_REPO_ID,
                   'FINETUNE_CHECKPOINT_HF_REPO_ID': FINETUNE_CHECKPOINT_HF_REPO_ID,
                   'FINETUNE_CHECKPOINT_HF_REPO_TYPE': 'model',
                   'FINETUNE_CHECKPOINT_FILENAME': FINETUNE_CHECKPOINT_FILENAME})
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['PYTHON_EXECUTABLE'] = sys.executable
REPO_SRC = '/kaggle/working/Efficient_VLM_For_Autonomous_Driving/src'
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
os.environ['PYTHONPATH'] = REPO_SRC
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception as exc:
    print('HF_TOKEN was not loaded:', exc)
print(json.dumps({'branch': GITHUB_BRANCH, 'config': os.environ['CONFIG'],
                  'visual_tokens': 49, 'gradient_checkpointing': False,
                  'hf_token_present': bool(os.environ.get('HF_TOKEN'))}, indent=2))


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving/.git ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
if ! "$PYTHON_EXECUTABLE" -m pip install --only-binary=:all: tokenizers==0.20.4; then
  echo "No compatible tokenizers==0.20.4 wheel. Select a Kaggle Python environment with a supported wheel; source builds are disabled." >&2
  exit 1
fi
"$PYTHON_EXECUTABLE" -m pip install --only-binary=tokenizers -e ".[mvpruner]" transformers==4.46.3 accelerate==1.15.0
"$PYTHON_EXECUTABLE" -m pip install -q pycocoevalcap
"$PYTHON_EXECUTABLE" - <<'PY'
import os, torch, tokenizers, transformers, accelerate
from efficient_vlm_ad.config import load_config
assert transformers.__version__ == "4.46.3", transformers.__version__
assert tokenizers.__version__ == "0.20.4", tokenizers.__version__
assert accelerate.__version__ == "1.15.0", accelerate.__version__
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
assert torch.cuda.device_count() >= 2, 'Select Kaggle 2 GPUs'
cfg = load_config(os.environ['CONFIG'])
assert cfg.runtime.precision == 'fp32'
assert cfg.model.vision.name == 'dinov2_small_avgpool'
assert cfg.model.text.model_id == 'google/t5-efficient-mini'
assert cfg.model.fusion.name == 'mvpruner_camera_query'
assert cfg.training.entropy_weight == 0.5
assert cfg.training.gradient_checkpointing is False
print('FP32', torch.__version__, transformers.__version__, tokenizers.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
PY
nvidia-smi


In [ ]:
import json, math, os, shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
REPO_SRC = '/kaggle/working/Efficient_VLM_For_Autonomous_Driving/src'
if REPO_SRC not in sys.path:
    sys.path.insert(0, REPO_SRC)
sys.path_importer_cache.pop(REPO_SRC, None)
from efficient_vlm_ad.config import load_config
from efficient_vlm_ad.cache import dinov2_manifest_fields, validate_manifest
from efficient_vlm_ad.modeling.vision import dinov2_processor_policy

def download_exact(repo_id, repo_type, destination, filenames):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source = hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=filename)
        shutil.copy2(source, destination / filename)

def validate_prepared(prepared, full=False, expected_counts=None):
    prepared = Path(prepared)
    manifest = json.loads((prepared / 'manifest.json').read_text(encoding='utf-8'))
    if full and manifest.get('subset') != 'full':
        raise ValueError('Prepared data must be the full subset')
    counts = {}
    for split in ('train', 'val', 'test'):
        with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
            counts[split] = sum(1 for line in stream if line.strip() and json.loads(line))
        if counts[split] <= 0 or counts[split] != manifest['counts'][split]:
            raise ValueError(f'Prepared data count mismatch: {split}')
    if full and expected_counts and counts != expected_counts:
        raise ValueError(f'Prepared counts do not match expected dataset counts: {counts}')
    print('Prepared data counts:', counts)
    return manifest

def validate_cache(cfg):
    cache = Path(cfg.cache.dir)
    manifest = json.loads((cache / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['version'] == 2, 'DINO cache requires manifest v2'
    index = json.loads((cache / 'index.json').read_text(encoding='utf-8'))
    shape = manifest['shape']
    if len(shape) != 4 or shape[1:] != [6, 64, 384] or shape[0] != len(index) or shape[0] <= 0:
        raise ValueError(f'Expected DINO feature shape [N,6,64,384], got {shape}')
    if set(index.values()) != set(range(shape[0])):
        raise ValueError('Cache index must map every feature row exactly once')
    expected = dinov2_manifest_fields(cfg, shape, dinov2_processor_policy())
    validated = validate_manifest(cache / 'manifest.json', expected)
    if (cache / 'features.float16.memmap').stat().st_size != math.prod(shape) * 2:
        raise ValueError('Feature memmap byte length does not match manifest')
    prepared = Path(cfg.project.output_dir) / 'prepared_data'
    subset = json.loads((prepared / 'manifest.json').read_text(encoding='utf-8'))['subset']
    limit = None if subset == 'full' else int(cfg.training.max_steps or 64)
    for split in ('train', 'val', 'test'):
        with (prepared / f'{split}.jsonl').open(encoding='utf-8') as stream:
            for number, line in enumerate(line for line in stream if line.strip()):
                if limit is not None and number >= limit:
                    break
                record = json.loads(line)
                key = '|'.join(record['camera_paths'][camera] for camera in sorted(record['camera_paths']))
                if key not in index:
                    raise ValueError(f'Cache is missing {split} sample {number}')
    print('Validated DINO cache:', shape, validated.fingerprint)
    return validated

os.chdir('/kaggle/working/Efficient_VLM_For_Autonomous_Driving')
import yaml
TRACKED_CONFIG = Path('configs/dinov2_t5_efficient_mini_kaggle_2gpu_camera_query.yaml')
RUNTIME_CONFIG = Path('/kaggle/working/dinov2_camera_query_runtime.yaml')
raw = yaml.safe_load(TRACKED_CONFIG.read_text(encoding='utf-8'))
assert raw['training']['gradient_checkpointing'] is False, 'Camera-query gradient checkpointing is unsupported'
assert raw['training']['entropy_weight'] == 0.5
assert raw['training']['seed'] == 42
RUNTIME_CONFIG.write_text(yaml.safe_dump(raw, sort_keys=False), encoding='utf-8')
os.environ['CONFIG'] = str(RUNTIME_CONFIG)
cfg = load_config(RUNTIME_CONFIG)
assert cfg.model.fusion.name == 'mvpruner_camera_query'
print(json.dumps({'config': str(RUNTIME_CONFIG), 'visual_tokens': 49,
                  'gradient_checkpointing': cfg.training.gradient_checkpointing,
                  'output_dir': cfg.project.output_dir}, indent=2))


In [ ]:
prepared = Path(cfg.project.output_dir) / 'prepared_data'
cache = Path(cfg.cache.dir)
prepared_files = ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl')
cache_files = ('features.float16.memmap', 'index.json', 'manifest.json')
if not all((cache / name).is_file() for name in cache_files):
    if CACHE_HF_REPO_ID:
        download_exact(CACHE_HF_REPO_ID, 'dataset', cache, cache_files)
    else:
        print('DINO cache is missing: materializing full raw images and refreshing prepared data before feature extraction. Prepared JSON alone may reference another machine.')
        subprocess.run([sys.executable, '-m', 'efficient_vlm_ad', 'prepare-data', '--config', os.environ['CONFIG'], '--subset', 'full'], check=True)
        validate_prepared(prepared, full=True, expected_counts=cfg.data.expected_counts)
        subprocess.run([sys.executable, '-m', 'efficient_vlm_ad', 'prepare-features', '--config', os.environ['CONFIG'], '--subset', 'full'], check=True)
if not all((prepared / name).is_file() for name in prepared_files):
    if PREPARED_DATA_HF_REPO_ID:
        download_exact(PREPARED_DATA_HF_REPO_ID, 'dataset', prepared, prepared_files)
    else:
        subprocess.run([sys.executable, '-m', 'efficient_vlm_ad', 'prepare-data', '--config', os.environ['CONFIG'], '--subset', 'full'], check=True)
validate_prepared(prepared, full=True, expected_counts=cfg.data.expected_counts)
validate_cache(cfg)


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
"$PYTHON_EXECUTABLE" -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage finetune --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
mkdir -p "$PROFILE/checkpoints"
RESUME_PATH=""
LOCAL_LATEST="$PROFILE/checkpoints/finetune_latest.pt"
if [ -f "$LOCAL_LATEST" ]; then
  RESUME_PATH="$LOCAL_LATEST"
elif [ -n "$FINETUNE_CHECKPOINT_HF_REPO_ID" ]; then
  "$PYTHON_EXECUTABLE" - "$FINETUNE_CHECKPOINT_HF_REPO_ID" "$FINETUNE_CHECKPOINT_HF_REPO_TYPE" "$PROFILE/checkpoints" "$FINETUNE_CHECKPOINT_FILENAME" <<'PY'
import sys
from huggingface_hub import snapshot_download
repo, kind, destination, filename = sys.argv[1:]
snapshot_download(repo_id=repo, repo_type=kind, local_dir=destination,
                  allow_patterns=[filename, filename + '.rng/**'])
PY
  RESUME_PATH="$PROFILE/checkpoints/$FINETUNE_CHECKPOINT_FILENAME"
else
  echo 'No finetune checkpoint configured; starting finetune from pretrained T5 and newly initialized multimodal modules.'
fi
RESUME=()
if [ -n "$RESUME_PATH" ]; then
  test -f "$RESUME_PATH" || { echo "Checkpoint is missing: $RESUME_PATH" >&2; exit 1; }
  test -f "$RESUME_PATH.rng/manifest.json" || { echo 'Distributed RNG manifest is missing' >&2; exit 1; }
  test -f "$RESUME_PATH.rng/rank_00000.pt" || { echo 'RNG for rank 0 is missing' >&2; exit 1; }
  test -f "$RESUME_PATH.rng/rank_00001.pt" || { echo 'RNG for rank 1 is missing' >&2; exit 1; }
  "$PYTHON_EXECUTABLE" - "$RESUME_PATH.rng/manifest.json" <<'PY'
import json, sys
manifest = json.load(open(sys.argv[1]))
assert manifest['world_size'] == 2, 'Resume RNG world size must be 2'
PY
  "$PYTHON_EXECUTABLE" -m efficient_vlm_ad verify-resume-checkpoint --config "$CONFIG" --stage finetune --checkpoint "$RESUME_PATH" --debug --debug-samples 1 --debug-numerics
  RESUME=(--resume "$RESUME_PATH")
fi
PYTHONUNBUFFERED=1 "$PYTHON_EXECUTABLE" -m accelerate.commands.launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune "${RESUME[@]}" --debug --debug-samples 1 --debug-numerics
tail -n 30 "$PROFILE/debug/train_progress.jsonl"


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
if [ ! -f "$CHECKPOINT" ]; then CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"; fi
"$PYTHON_EXECUTABLE" -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" --batch-size 16
"$PYTHON_EXECUTABLE" -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT"
cat "$PROFILE/metrics.json"
cat "$PROFILE/benchmark.json"
head -n 8 "$PROFILE/predictions.jsonl"


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
ARCHIVE=/kaggle/working/dinov2_camera_query_finetune_artifacts.tar.gz
tar -czf "$ARCHIVE" "$CONFIG" "$PROFILE/checkpoints" "$PROFILE/metrics.json" "$PROFILE/benchmark.json" "$PROFILE/predictions.jsonl" "$PROFILE/debug"
ls -lh "$ARCHIVE"
